# 2b, $\{f, g\} = \omega(X_f, X_g) = X_f(g) = \iota_{X_f} dg = \iota_{X_f}\iota_{X_g}\omega$

**Problem (b).** In the same symplectic setup ($d\omega = 0$,
$\iota_{X_f}\omega = df$, $\{f,g\} := \pi(df, dg)$,
$\pi = \omega^{-1}$), close the five-term chain:

$$
\{f, g\}
\;\underset{(1)}{=}\; \omega(X_f, X_g)
\;\underset{(2)}{=}\; X_f(g)
\;\underset{(3)}{=}\; \iota_{X_f} dg
\;\underset{(4)}{=}\; \iota_{X_f}\iota_{X_g}\omega.
$$

Since the five terms form a chain, end-to-end this is a single
equality: $\{f, g\} = X_f(g)$. The notebook closes that, and the
intermediate terms appear automatically **as intermediate values in
the proof chain**.

## Status of the equalities in the chain

Our system currently knows the form **at the grading level** (a
degree-2 graded symbol); there is **no** intrinsic 2-form evaluation
node like `ω(X, Y)` (see the Phase 12 outline). So the four
equalities in the chain have different statuses:

| # | Equality | Status | Source |
|---|---|---|---|
| (1) | $\{f,g\} = \omega(X_f, X_g)$ | Definitional (problem-given: $\pi = \omega^{-1}$) | Inline axiom A1 |
| (2) | $\omega(X_f, X_g) = \iota_{X_f}\iota_{X_g}\omega$ | Definitional (2-form evaluation convention) | Inline axiom A2 |
| (3) | $\iota_{X_g}\omega = dg$ | Hamiltonian defining relation | Inline axiom A3 (the $g$-counterpart of 2a's) |
| (4) | $\iota_X(df) = X(f)$ | **Built-in pairing rule** | `IotaOnExactOneFormDefinition` (default engine) |

When Phase 12 lands, (1) and (2) will move from **axiom** to
**theorem**, once tensor-eval is intrinsic, $\omega(X,Y) \to
\iota_X\iota_Y\omega$ will derive automatically and the
musical-inverse bilinear coupling will validate
$\pi(df, dg) = \omega(X_f, X_g)$. A "tightened" revision of this
notebook will follow at that point.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Setup

- $f, g$, functions (0-forms).
- $\omega$, symplectic 2-form (Graded(2)).
- $X_f, X_g$, Hamiltonian vector fields (degree-0 Derivations).
- `poisson_fg` (`{f,g}`) and `omega_XfXg` (`ω(X_f,X_g)`), since we
  have no intrinsic node, we declare them as **named Symbols**; the
  problem statement's definitional identifications are added as
  axioms below.
- Both composites (Poisson bracket and 2-form eval) are degree 0 in
  classical Poisson calculus: their values are scalar functions.

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.core.expr import Expr, Integer, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
g = Symbol("g")
reg.declare(f, Graded(degree=0))
reg.declare(g, Graded(degree=0))

omega = Symbol("ω")
reg.declare(omega, Graded(degree=2))

X_f = Derivation("X_f", degree=0)
X_g = Derivation("X_g", degree=0)

# Named composites, system treats them as scalar (degree-0) symbols
# and the axioms below define what they equal.
poisson_fg = Symbol("{f,g}")
omega_XfXg = Symbol("ω(X_f,X_g)")
reg.declare(poisson_fg, Graded(degree=0))
reg.declare(omega_XfXg, Graded(degree=0))

print("f, g           :", f, g)
print("ω              :", omega)
print("X_f, X_g       :", X_f, X_g)
print("poisson_fg     :", poisson_fg)
print("omega_XfXg     :", omega_XfXg)

f, g           : f g
ω              : ω
X_f, X_g       : X_f X_g
poisson_fg     : {f,g}
omega_XfXg     : ω(X_f,X_g)


## 2. Four problem axioms

All four are `Definition`s of the form "if you see the LHS pattern,
replace it with the RHS":

- **A1**, Poisson bracket definition:
  $\{f,g\} = \omega(X_f, X_g)$. A direct consequence of the
  problem's $\pi = \omega^{-1}$ assumption.
- **A2**, 2-form evaluation convention:
  $\omega(X_f, X_g) = \iota_{X_f}\iota_{X_g}\omega$. Fills the
  intrinsic gap in our system.
- **A3**, Hamiltonian defining relation (g):
  $\iota_{X_g}\omega = dg$. The $g$-counterpart of 2a's A2.
- **A4**, Hamiltonian defining relation (f):
  $\iota_{X_f}\omega = df$. **Technically not needed** for this
  proof since the closure for $X_f(g)$ falls directly to pairing,
  but we add it to fully reflect the problem's givens.

In [3]:
class PoissonDef(Definition):
    """A1: {f,g} = ω(X_f, X_g)."""
    name = "{f,g} = ω(X_f, X_g)"

    def matches(self, expr):
        return expr == poisson_fg

    def rewrite(self, expr):
        return omega_XfXg


class FormEvalDef(Definition):
    """A2: ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω."""
    name = "ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω"

    def matches(self, expr):
        return expr == omega_XfXg

    def rewrite(self, expr):
        return Act(interior(X_f), Act(interior(X_g), omega))


class IotaXgOmegaIsDg(Definition):
    """A3: ι_{X_g} ω = dg."""
    name = "ι_{X_g} ω = dg"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_g:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, g)


class IotaXfOmegaIsDf(Definition):
    """A4: ι_{X_f} ω = df (completeness; not needed for this proof)."""
    name = "ι_{X_f} ω = df"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_f:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, f)


for axiom in (PoissonDef, FormEvalDef, IotaXgOmegaIsDg, IotaXfOmegaIsDf):
    print("-", axiom.name)

- {f,g} = ω(X_f, X_g)
- ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω
- ι_{X_g} ω = dg
- ι_{X_f} ω = df


## 3. Engine

`default_engine` already carries the pairing rule
(**$\iota_X(df) = X(f)$**). We add the four problem axioms on top.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(PoissonDef())
engine.register(FormEvalDef())
engine.register(IotaXgOmegaIsDg())
engine.register(IotaXfOmegaIsDf())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 12 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - {f,g} = ω(X_f, X_g)
 - ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω
 - ι_{X_g} ω = dg
 - ι_{X_f} ω = df


## 4. Goal and proof

End-to-end equality of the chain: $\{f, g\} = X_f(g)$. The engine
will fire definitions on this obstruction to a fixed point;
bottom-up firing means the order is:
1. A1, `{f,g} → ω(X_f,X_g)`
2. A2, `ω(X_f,X_g) → ι_{X_f}(ι_{X_g}(ω))`
3. A3, innermost first: `ι_{X_g}(ω) → dg`
4. Pairing (built-in), `ι_{X_f}(dg) → X_f(g)`
5. Simplify, `X_f(g) + (−X_f(g)) → 0`

In [5]:
lhs = poisson_fg
rhs = Act(X_f, g)

print("LHS:", lhs)
print("RHS:", rhs)

chain = ExpandAndSimplify().prove(
    lhs, rhs, registry=reg, engine=engine
)
print(f"\nCLOSED, {len(chain)} steps.")


LHS: {f,g}
RHS: X_f(g)

CLOSED, 5 steps.


## 5. Proof chain, LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

{\allowdisplaybreaks\scriptsize
\begin{gather*}
{f,g} \to \omega(X_{f,X_g)} \quad \text{[{f,g} = \ensuremath{\omega}(X\_f, X\_g)]\,(axiom)} \\
\omega(X_{f,X_g)} \to \iota_{X_f}\!\left(\iota_{X_g}\!\left(\omega\right)\right) \quad \text{[\ensuremath{\omega}(X\_f, X\_g) = \ensuremath{\iota}\_{X\_f} \ensuremath{\iota}\_{X\_g} \ensuremath{\omega}]\,(axiom)} \\
\iota_{X_g}\!\left(\omega\right) \to d\!\left(g\right) \quad \text{[\ensuremath{\iota}\_{X\_g} \ensuremath{\omega} = dg]\,(axiom)} \\
\iota_{X_f}\!\left(d\!\left(g\right)\right) \to X_f\!\left(g\right) \quad \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)} \\
X_f\!\left(g\right) - X_f\!\left(g\right) \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{gather*}
}

## 6. Step by step

This chain is essentially the same five-term chain from the problem
statement, the intermediate expressions naturally surface as
intermediate values.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()


[1] {f,g} = ω(X_f, X_g) [axiom]
    {f,g}
 -> ω(X_f,X_g)

[2] ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω [axiom]
    ω(X_f,X_g)
 -> ι_X_f(ι_X_g(ω))

[3] ι_{X_g} ω = dg [axiom]
    ι_X_g(ω)
 -> d(g)

[4] ι_X(df) = X(f) [axiom]
    ι_X_f(d(g))
 -> X_f(g)

[5] simplify 
    (X_f(g) + (-X_f(g)))
 -> 0



## 7. Show the chain in problem-statement format

We list the five terms in the order of the problem statement. Each
intermediate term is the name of an intermediate expression in the
chain:

In [8]:
chain_terms = [
    (poisson_fg,                              "problem definition"),
    (omega_XfXg,                              "A1: {f,g} = ω(X_f, X_g)"),
    (Act(interior(X_f), Act(interior(X_g), omega)),
                                              "A2: ω(X_f, X_g) = ι_{X_f}ι_{X_g}ω"),
    (Act(interior(X_f), Act(d, g)),           "A3: ι_{X_g}ω = dg"),
    (Act(X_f, g),                             "built-in: ι_X(df) = X(f)"),
]

print("{f,g}")
for term, reason in chain_terms[1:]:
    print(f"   =  {term}   [{reason}]")


{f,g}
   =  ω(X_f,X_g)   [A1: {f,g} = ω(X_f, X_g)]
   =  ι_X_f(ι_X_g(ω))   [A2: ω(X_f, X_g) = ι_{X_f}ι_{X_g}ω]
   =  ι_X_f(d(g))   [A3: ι_{X_g}ω = dg]
   =  X_f(g)   [built-in: ι_X(df) = X(f)]


## Conclusion

$$\boxed{\;\{f, g\} = X_f(g)\;}$$

closed in a 5-step chain. The five-term display in the problem
statement is precisely a readout of the intermediate values in the
proof chain.

Status of the axioms in this notebook:

- **A3, A4**, genuine mathematical givens (the Hamiltonian defining
  relation); will remain axioms after Phase 12 too.
- **A1, A2**, currently definitional axioms; once Phase 12 lands
  with intrinsic tensor-eval, they will move to **theorems**.
- **Pairing $\iota_X(df) = X(f)$**, already built-in; unchanged.

Together with the result $\mathcal{L}_{X_f}\omega = 0$ from 2a, this
chain closes the basic computational identities of the Poisson
bracket (subsequent identities like antisymmetry, Leibniz, the Jacobi
chain follow the same pattern, extra axiom set + the same engine).